# ThermoRoute v3: complete implementation (Colab or Jupyter)

Thermal throttling aware expert scheduling for smartphone MoE inference (OLMoE 1B 7B).

**Run every cell from top to bottom.** Each step saves its output to disk and is skipped on a
re run, so a Colab disconnect only costs the step that was running.

| Step | What it produces | Runtime | Hardware |
|---|---|---|---|
| 0 | settings, packages, fonts, HuggingFace login | 3 to 5 min | any |
| 1 | **measured** OLMoE perplexity per k, router expert union, config | 20 to 40 min | **GPU** (Colab T4) |
| 2 | **real** LMSYS Chat 1M request lengths (OLMoE tokenizer) | 5 to 15 min | any |
| 3 | sanity checks of the simulator | 1 min | CPU |
| 4 | MPC weight tuning (training seeds), FiDRL training, full experiment grid | 30 to 60 min | CPU |
| 5 | tables (LaTeX), Wilcoxon with direction and Holm correction | 1 min | CPU |
| 6 | 11 figures, PNG 800 dpi and PDF | 2 min | CPU |
| 7 | claim check against the abstract, zip and download | 1 min | CPU |

**Before Step 1:** Colab menu Runtime, Change runtime type, T4 GPU. Step 1 checks its own outputs and redoes only what is missing or invalid.
**Before Step 2:** open https://huggingface.co/datasets/lmsys/lmsys-chat-1m while logged in and accept
the terms, then create a READ token at https://huggingface.co/settings/tokens.

**What is real, derived and assumed (state this in the paper):**
1. Measured: OLMoE perplexity for k in {2,4,6,8} (WikiText 103), distinct experts per layer over
   consecutive tokens (router logits), parameter counts (config.json).
2. Real: prompt and response lengths of LMSYS Chat 1M conversations, OLMoE tokenizer.
3. Derived: phone decode speed from a memory bound model (bandwidth / bytes per token).
4. Representative: thermal R and C, power coefficients, governor thresholds, SoC profiles.
5. Assumed: think time gaps, prefill multiplier, speculative acceptance rate, speed target, SLO.
6. This is a calibrated simulation of two SoC classes, not measurements on commercial phones.

In [ ]:
# ===== Step 0a: settings =====
USE_DRIVE = True        # Colab: keep everything in Google Drive so a disconnect loses nothing
import os, sys
IN_COLAB = "google.colab" in sys.modules
if os.environ.get("TR_ROOT"):
    ROOT = os.environ["TR_ROOT"]
elif IN_COLAB and USE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/ThermoRoute"
elif IN_COLAB:
    ROOT = "/content/ThermoRoute"
else:
    ROOT = os.path.abspath(".") if os.path.isdir("sim") else os.path.abspath("ThermoRoute_work")
os.makedirs(os.path.join(ROOT, "sim"), exist_ok=True)
os.chdir(ROOT)
os.environ["TR_ROOT"] = ROOT
os.environ.setdefault("TR_FIG_DIR", os.path.join(ROOT, "Downloads", "ThermoRoute") if IN_COLAB
                      else os.path.join(os.path.expanduser("~"), "Downloads", "ThermoRoute"))
sys.path.insert(0, os.path.join(ROOT, "sim"))
print("working folder:", ROOT, "\nfigures folder:", os.environ["TR_FIG_DIR"])

In [ ]:
# ===== Step 0b: packages (transformers pinned: OLMoE experts must stay nn.Linear for 8 bit loading)
%pip install -q "transformers==4.51.3" accelerate bitsandbytes datasets huggingface_hub scipy pandas matplotlib

In [ ]:
# ===== Step 0c: Times New Roman for figures (Colab only; Windows already has it)
if IN_COLAB:
    !echo ttf-mscorefonts-installer msttcorefonts/accepted-mscorefonts-eula select true | debconf-set-selections
    !apt-get -qq install -y ttf-mscorefonts-installer fonts-liberation > /dev/null 2>&1
    !fc-list | grep -i -c "times"

In [ ]:
# ===== Step 0d: HuggingFace login (needed for LMSYS Chat 1M, which is gated)
from huggingface_hub import login
if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"])
else:
    login()       # paste your READ token

## Simulator source files
These cells write the simulator package to `sim/`. Edit constants only in `config.py`.

In [ ]:
%%writefile sim/config.py
"""ThermoRoute v3: single source of truth for every constant.
Tags: [measured] produced by Step 1 / Step 2 of the notebook, [spec] vendor specification (verify),
[repr] representative value declared in the paper, [assumed] modelling assumption."""
import os

ROOT = os.environ.get("TR_ROOT", os.getcwd())
DATA_DIR = os.path.join(ROOT, "data")
RESULTS_DIR = os.path.join(ROOT, "results")
TRACE_DIR = os.path.join(RESULTS_DIR, "traces")
FIG_DIR = os.environ.get("TR_FIG_DIR",
                         os.path.join(os.path.expanduser("~"), "Downloads", "ThermoRoute"))
for _d in (DATA_DIR, RESULTS_DIR, TRACE_DIR, FIG_DIR):
    os.makedirs(_d, exist_ok=True)

# Experiment grid
SEEDS_TEST = list(range(20))            # 20 seeds: Holm corrected Wilcoxon can reach p < 0.05
SEEDS_TRAIN = list(range(100, 120))           # FiDRL training only, disjoint from test seeds
SIM_SECONDS = 1800
AMBIENTS = [25.0, 30.0, 35.0]
WORKLOADS = {"chat": (3.0, 12.0), "agent": (0.5, 2.0)}   # think time range in s [assumed]
MAIN_COND = dict(soc="SoC-A", amb=25.0, wl="agent")      # main table
STRESS_COND = dict(soc="SoC-A", amb=35.0, wl="agent")    # sweeps run here (constraints active)
N_JOBS = max(1, (os.cpu_count() or 2))
if os.environ.get("THERMOROUTE_QUICK"):                  # smoke test only, never for the paper
    SEEDS_TEST, SEEDS_TRAIN, SIM_SECONDS = [0, 1], [100], 300

# Model (OLMoE 1B 7B: 64 experts, top 8)
HF_MODEL = "allenai/OLMoE-1B-7B-0924"
K_LEVELS = [2, 4, 6, 8]
BPP = 0.56                     # bytes per parameter, Q4_K_M style on device weights [assumed]

# DVFS and throughput
F_LEVELS = [1.0, 0.85, 0.70, 0.55, 0.40, 0.30]
CORES = ["performance", "efficiency"]
TPS_FREQ_EXP = 0.7             # memory bound decode, achieved bandwidth ~ f^0.7 [assumed]
PREFILL_MULT = 8.0             # prefill tokens/s relative to decode tokens/s [assumed]
SLO_TPS = 20.0                 # per response decode speed SLO, tok/s [assumed]
BATTERY_J = 19.25 * 3600.0     # 5000 mAh at 3.85 V

# SoC profiles. Power = idle + compute_w[core] f^3 + e_byte x bytes/s moved.
SOCS = {
    "SoC-A": dict(label="Flagship class", mem_bw_gbs=67.2, bw_eff=0.35,       # mem_bw [spec]
                  idle_w=0.5, e_byte=8.0e-11,
                  compute_w={"performance": 4.5, "efficiency": 1.6},
                  core_eff={"performance": 1.0, "efficiency": 0.55},
                  Cd=4.5, Rds=4.5, Cs=85.0, Rsa=3.6),
    "SoC-B": dict(label="Mid range class", mem_bw_gbs=51.2, bw_eff=0.35,
                  idle_w=0.4, e_byte=8.0e-11,
                  compute_w={"performance": 3.6, "efficiency": 1.3},
                  core_eff={"performance": 1.0, "efficiency": 0.55},
                  Cd=4.0, Rds=5.0, Cs=75.0, Rsa=3.9),
}

# OS thermal governor, identical for every method [repr]. Progressive step governor:
# every GOV_PERIOD s the cap moves one F_LEVELS step down while above the trip point and
# one step up once below the release point. Die emergency cap at T_CRIT.
GOV_PERIOD = 2
T_TRIP, T_RELEASE, T_CRIT = 85.0, 80.0, 95.0
T_SKIN, T_SKIN_RELEASE = 43.0, 42.0

# Controller side uncertainty
SENSOR_NOISE = 0.5             # C, std
MISMATCH_DEFAULT = 0.10        # +/- uniform fraction on R and C of the true plant

# ThermoRoute MPC defaults
MPC = dict(H=10, L=180, margin=3.0, skin_margin=1.0, lam=5.0, lam_s=5.0, lam_T=5.0,
           mu=1.0, tps_target=30.0, wq=10.0, ws=6.0, we=2.0)
MPC_WEIGHTS_JSON = os.path.join(RESULTS_DIR, "mpc_weights.json")   # written by the tuning step
if os.path.exists(MPC_WEIGHTS_JSON):
    import json as _json
    MPC.update(_json.load(open(MPC_WEIGHTS_JSON)))
TUNE_GRID = dict(wq=[10.0, 20.0, 40.0], we=[0.5, 1.0, 2.0], ws=[6.0, 12.0])
TUNE_CONDS = [("SoC-A", 25.0, "agent"), ("SoC-A", 35.0, "agent"), ("SoC-A", 35.0, "chat")]

# Speculative decoding (EdgeLLM style) [assumed]; expert union per verify step is measured
SPEC = dict(alpha=0.7, gamma=4, draft_cost=0.10, extra_w=0.8)

# Sweeps (run on STRESS_COND)
SWEEP_H = [5, 10, 20, 40]
SWEEP_M = [0.5, 1.0, 2.0]      # skin margin (the skin limit binds; the die limit does not)
SWEEP_LAM = [0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0]
SWEEP_MISMATCH = [0.0, 0.1, 0.2, 0.3]

METHODS = ["EdgeMoE", "EdgeLLM", "ReactiveDVFS", "FiDRL", "StaticK4",
           "ThermoRoute", "ThermoRoute+Spec", "Oracle"]

In [ ]:
%%writefile sim/model_profile.py
"""OLMoE parameter counts (from config.json saved in Step 1), measured quality profile Q(k),
and measured expert union per speculative verify window."""
import os, json
import numpy as np, pandas as pd
from config import DATA_DIR, K_LEVELS, SPEC

def _need(name):
    path = os.path.join(DATA_DIR, name)
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} missing. Run Step 1 of the notebook first.")
    return path

def load_olmoe_config():
    with open(_need("olmoe_config.json")) as f:
        return json.load(f)

def param_counts():
    """Parameters read from memory per decoded token: shared part and one expert slot."""
    c = load_olmoe_config()
    h, i, L = c["hidden_size"], c["intermediate_size"], c["num_hidden_layers"]
    E, V = c["num_experts"], c["vocab_size"]
    nh = c["num_attention_heads"]
    nkv = c.get("num_key_value_heads") or nh
    hd = h // nh
    attn = 2 * h * h + 2 * h * nkv * hd                  # q, o, k, v projections
    shared = L * (attn + h * E) + V * h                  # attention + router + lm_head
    expert = L * 3 * h * i                               # one active expert slot, all layers
    return float(shared), float(expert), int(E)

def load_quality():
    """dict k -> (Q, ppl) with Q(8) = 1, from data/k_profile.csv (measured)."""
    df = pd.read_csv(_need("k_profile.csv")).set_index("k")
    ref = df.loc[max(K_LEVELS), "ppl"]
    return {k: (float(ref / df.loc[k, "ppl"]), float(df.loc[k, "ppl"])) for k in K_LEVELS}

def load_union():
    """dict k -> mean number of distinct experts per layer over a window of gamma+1
    consecutive tokens (measured in Step 1 from OLMoE router logits)."""
    df = pd.read_csv(_need("expert_union.csv"))
    df = df[df.window == SPEC["gamma"] + 1].set_index("k")
    return {k: float(df.loc[k, "union"]) for k in K_LEVELS}

In [ ]:
%%writefile sim/plant.py
"""Two node (die, skin) RC thermal plant, memory bound throughput and power models,
and the progressive OS thermal governor shared by every method."""
import numpy as np
from config import (TPS_FREQ_EXP, BPP, F_LEVELS, GOV_PERIOD, T_TRIP, T_RELEASE, T_CRIT,
                    T_SKIN, T_SKIN_RELEASE, SENSOR_NOISE, SPEC)
from model_profile import param_counts, load_union

SHARED_P, EXPERT_P, N_EXPERTS = param_counts()
UNION = load_union()
ACC = (1 - SPEC["alpha"] ** (SPEC["gamma"] + 1)) / (1 - SPEC["alpha"])   # expected tokens/step

def _core(p, key, core):
    return np.vectorize(lambda c: p[key][c])(core) if np.ndim(core) else p[key][core]

def bandwidth(p, f, core):
    """Bytes/s moved while decoding (memory bound)."""
    return p["mem_bw_gbs"] * 1e9 * p["bw_eff"] * np.asarray(f) ** TPS_FREQ_EXP \
        * _core(p, "core_eff", core)

def bytes_per_token(k, spec=False):
    """Plain decode: shared + k expert slots. Speculative decode: gamma draft passes plus one
    verify pass that reads the union of experts selected by the gamma+1 verified tokens."""
    k = np.asarray(k, float)
    plain = BPP * (SHARED_P + k * EXPERT_P)
    if not spec:
        return plain
    u = np.vectorize(lambda kk: UNION[int(kk)])(k) if np.ndim(k) else UNION[int(k)]
    step = SPEC["gamma"] * SPEC["draft_cost"] * plain + BPP * (SHARED_P + u * EXPERT_P)
    return step / ACC

def tps(p, k, f, core, spec=False):
    return bandwidth(p, f, core) / bytes_per_token(k, spec)

def power_active(p, k, f, core, spec=False):
    """At a given frequency the memory system is saturated, so k changes tokens per byte,
    not watts. Fewer experts finish a response sooner and lengthen idle time."""
    f = np.asarray(f)
    comp = (_core(p, "compute_w", core) + (SPEC["extra_w"] if spec else 0.0)) * f ** 3
    return p["idle_w"] + comp + p["e_byte"] * bandwidth(p, f, core) + 0 * np.asarray(k, float)

def thermal_step(Td, Ts, P, p, T_amb, dt=1.0):
    Td_n = Td + dt / p["Cd"] * (P - (Td - Ts) / p["Rds"])
    Ts_n = Ts + dt / p["Cs"] * ((Td - Ts) / p["Rds"] - (Ts - T_amb) / p["Rsa"])
    return Td_n, Ts_n

class Plant:
    def __init__(self, soc, T_amb, rng, mismatch):
        self.p = {k: (dict(v) if isinstance(v, dict) else v) for k, v in soc.items()}
        for key in ("Cd", "Rds", "Cs", "Rsa"):
            self.p[key] *= 1 + rng.uniform(-mismatch, mismatch)
        self.T_amb, self.rng = T_amb, rng
        self.Td = self.Ts = T_amb
        self.jd = self.js = 0                 # governor step index into F_LEVELS
        self.tick = 0

    @property
    def gov_die(self): return self.jd > 0
    @property
    def gov_skin(self): return self.js > 0

    def cap(self):
        return min(F_LEVELS[self.jd], F_LEVELS[self.js])

    def step(self, P):
        self.Td, self.Ts = thermal_step(self.Td, self.Ts, P, self.p, self.T_amb)
        self.tick += 1
        last = len(F_LEVELS) - 1
        if self.Td >= T_CRIT:
            self.jd = last
        elif self.tick % GOV_PERIOD == 0:
            if self.Td >= T_TRIP: self.jd = min(self.jd + 1, last)
            elif self.Td <= T_RELEASE: self.jd = max(self.jd - 1, 0)
        if self.tick % GOV_PERIOD == 0:
            if self.Ts >= T_SKIN: self.js = min(self.js + 1, last)
            elif self.Ts <= T_SKIN_RELEASE: self.js = max(self.js - 1, 0)

    def measure(self, noise=True):
        n = self.rng.normal(0, SENSOR_NOISE, 2) if noise else np.zeros(2)
        return self.Td + n[0], self.Ts + n[1]

In [ ]:
%%writefile sim/workload.py
"""Closed loop single user workload. Request lengths are real (LMSYS Chat 1M, OLMoE
tokenizer); think time gaps are synthetic because the dataset has no timestamps."""
import os, numpy as np, pandas as pd
from config import DATA_DIR, PREFILL_MULT

_REQ = None
def requests_table():
    global _REQ
    if _REQ is None:
        path = os.path.join(DATA_DIR, "requests.csv")
        if not os.path.exists(path):
            raise FileNotFoundError(f"{path} missing. Run Step 2 of the notebook first.")
        _REQ = pd.read_csv(path)[["prompt_tokens", "response_tokens"]].to_numpy(float)
    return _REQ

def workload_stats(think_range):
    """Offline profile available to the controller: mean prompt, response and think time."""
    r = requests_table()
    return float(r[:, 0].mean()), float(r[:, 1].mean()), float(np.mean(think_range))

class Workload:
    def __init__(self, think_range, rng):
        req = requests_table()
        self.req = req[rng.permutation(len(req))]
        self.rng, self.think_range = rng, think_range
        self.i, self.t = 0, 0.0
        self.think_left = rng.uniform(*think_range)
        self.cur = None
        self.done = []          # (latency, ttft, decode_tps)

    def busy(self):
        return self.cur is not None or self.think_left <= 0

    def _start(self):
        pt, rt = self.req[self.i % len(self.req)]; self.i += 1
        self.cur = dict(arr=self.t, prefill=float(pt), decode=float(rt), rt=float(rt), first=None)

    def advance(self, decode_tps, dt=1.0):
        """Run one second. Returns (active fraction, decoded tokens)."""
        budget, active, decoded = dt, 0.0, 0.0
        while budget > 1e-9:
            if self.cur is None:
                if self.think_left > 0:
                    w = min(budget, self.think_left)
                    self.think_left -= w; budget -= w
                    if self.think_left > 1e-12: break
                    self.think_left = 0.0
                self._start(); self.cur["arr"] = self.t + (dt - budget)
            c = self.cur
            if c["prefill"] > 0:
                need = c["prefill"] / (decode_tps * PREFILL_MULT)
                w = min(budget, need)
                c["prefill"] -= w * decode_tps * PREFILL_MULT
                budget -= w; active += w
                if c["prefill"] <= 1e-9:
                    c["prefill"] = 0.0; c["first"] = self.t + (dt - budget)
                continue
            need = c["decode"] / decode_tps
            w = min(budget, need)
            c["decode"] -= w * decode_tps; decoded += w * decode_tps
            budget -= w; active += w
            if c["decode"] <= 1e-9:
                end = self.t + (dt - budget)
                dec_t = max(end - c["first"], 1e-6)
                self.done.append((end - c["arr"], c["first"] - c["arr"], c["rt"] / dec_t))
                self.cur = None
                self.think_left = self.rng.uniform(*self.think_range)
        self.t += dt
        return active / dt, decoded

In [ ]:
%%writefile sim/controllers.py
"""Schedulers. Baselines are mechanism level reimplementations inside this simulator,
not the original authors' code. obs = dict(Td, Ts, cap, busy)."""
import itertools, numpy as np
from config import K_LEVELS, F_LEVELS, CORES, MPC, T_TRIP, T_SKIN, PREFILL_MULT
from plant import tps, power_active, thermal_step

FULL = max(K_LEVELS)

class Base:
    spec = False
    def __init__(self, soc, Q): self.soc, self.Q = soc, Q
    def act(self, obs): raise NotImplementedError
    def feedback(self, obs_new, a, P_applied): pass

class EdgeMoE(Base):          # full expert budget, thermally unaware, OS governor only
    def act(self, obs): return FULL, 1.0, "performance"

class StaticK4(Base):         # static expert pruning baseline
    def act(self, obs): return 4, 1.0, "performance"

class EdgeLLM(EdgeMoE):       # speculative decoding, thermally unaware
    spec = True

class ReactiveDVFS(Base):     # two threshold hysteresis on frequency
    def __init__(self, soc, Q):
        super().__init__(soc, Q); self.j = 0
    def act(self, obs):
        if obs["Td"] > T_TRIP - 5 or obs["Ts"] > T_SKIN - 0.5:
            self.j = min(self.j + 1, len(F_LEVELS) - 1)
        elif obs["Td"] < T_TRIP - 20 and obs["Ts"] < T_SKIN - 3:
            self.j = max(self.j - 1, 0)
        return FULL, F_LEVELS[self.j], "performance"

def fidrl_state(obs):
    d = int(np.clip((obs["Td"] - 25) // 10, 0, 7))
    s = int(np.clip((obs["Ts"] - 25) // 3, 0, 8))
    c = int(np.argmin(np.abs(np.array(F_LEVELS) - obs["cap"])))
    return d, s, int(obs["busy"]), c

class FiDRL(Base):            # tabular Q learning DVFS agent, trained on SEEDS_TRAIN
    def __init__(self, soc, Q, table=None, eps=0.0, rng=None):
        super().__init__(soc, Q)
        self.table = table if table is not None else np.zeros((8, 9, 2, len(F_LEVELS),
                                                               len(F_LEVELS)))
        self.eps, self.rng = eps, rng or np.random.default_rng(0)
        self.prev = None
    def act(self, obs):
        s = fidrl_state(obs)
        if self.rng.random() < self.eps: a = int(self.rng.integers(len(F_LEVELS)))
        else: a = int(np.argmax(self.table[s]))
        self.prev = (s, a)
        return FULL, F_LEVELS[a], "performance"
    def learn(self, obs_new, reward, lr=0.1, gamma=0.95):
        if self.prev is None: return
        s, a = self.prev; s2 = fidrl_state(obs_new)
        self.table[s + (a,)] += lr * (reward + gamma * self.table[s2].max() - self.table[s + (a,)])

class ThermoRoute(Base):
    """Rolling horizon MPC over (k, f, core).
    Stage cost: quality, speed up to a target, energy per token.
    Constraints (soft): die and skin margins over H steps, plus a lookahead terminal cost that
    projects skin temperature L seconds ahead with the skin time constant, so the controller
    lowers k before the skin limit is reached and restores it as the device cools.
    Duty cycle depends on the action through the closed loop workload profile."""
    def __init__(self, soc, Q, wl_stats, H=None, margin=None, lam=None, nominal=None,
                 T_amb=None, bias=True, w=None, skin_margin=None):
        super().__init__(soc, Q)
        c = dict(MPC); c.update(w or {})
        self.H = int(H or c["H"]); self.L = c["L"]
        self.m = c["margin"] if margin is None else margin
        self.lam = c["lam"] if lam is None else lam
        self.lam_s = c["lam_s"] if lam is None else lam
        self.lam_T = c["lam_T"] if lam is None else lam
        self.ms = c["skin_margin"] if skin_margin is None else skin_margin
        self.mu = c["mu"]
        self.tgt, self.wq, self.ws, self.we = c["tps_target"], c["wq"], c["ws"], c["we"]
        self.p = nominal or soc
        self.T_amb = T_amb                    # None: set from the first skin reading
        self.use_bias = bias
        self.Pbar, self.Rbar, self.Tbar = wl_stats
        self.acts = list(itertools.product(K_LEVELS, F_LEVELS, CORES))
        self.K = np.array([a[0] for a in self.acts], float)
        self.F = np.array([a[1] for a in self.acts], float)
        self.C = np.array([a[2] for a in self.acts])
        self.Qv = np.array([Q[int(k)][0] for k in self.K])
        self.E_ref = float(power_active(self.p, FULL, 1.0, "performance", self.spec)
                           / tps(self.p, FULL, 1.0, "performance", self.spec))
        self.tau_s = self.p["Rsa"] * self.p["Cs"]
        self.bd = self.bs = 0.0
        self.last = None
        self.Tdf = self.Tsf = None

    def _filter(self, obs):
        """First order filter on noisy sensor readings (die is fast, skin is slow)."""
        if self.Tdf is None:
            self.Tdf, self.Tsf = obs["Td"], obs["Ts"]
        else:
            self.Tdf = 0.5 * self.Tdf + 0.5 * obs["Td"]
            self.Tsf = 0.85 * self.Tsf + 0.15 * obs["Ts"]
        return self.Tdf, self.Tsf

    def act(self, obs):
        Tdm, Tsm = self._filter(obs)
        if self.T_amb is None:
            self.T_amb = Tsm                  # device starts in equilibrium with ambient
        cap = obs["cap"]
        f_eff = np.minimum(self.F, cap)
        Pa = power_active(self.p, self.K, f_eff, self.C, self.spec)
        tp = tps(self.p, self.K, f_eff, self.C, self.spec)
        Pi = self.p["idle_w"]
        serve = self.Pbar / (PREFILL_MULT * tp) + self.Rbar / tp
        d = serve / (serve + self.Tbar)                       # action dependent duty
        Td = np.full(len(self.K), Tdm); Ts = np.full(len(self.K), Tsm)
        pen = np.zeros(len(self.K))
        for h in range(self.H):
            a = 1.0 if (h == 0 and obs["busy"]) else d
            Td, Ts = thermal_step(Td, Ts, a * Pa + (1 - a) * Pi, self.p, self.T_amb)
            Td, Ts = Td + self.bd, Ts + self.bs
            pen += self.lam * np.maximum(0, Td - (T_TRIP - self.m)) ** 2
            pen += self.lam_s * np.maximum(0, Ts - (T_SKIN - self.ms)) ** 2
        P_avg = d * Pa + (1 - d) * Pi
        Ts_ss = self.T_amb + P_avg * self.p["Rsa"]
        Ts_L = Ts_ss + (Ts - Ts_ss) * np.exp(-self.L / self.tau_s) + self.bs
        Td_L = Ts_L + Pa * self.p["Rds"]
        pen += self.lam_T * (np.maximum(0, Ts_L - (T_SKIN - self.ms)) ** 2
                             + np.maximum(0, Td_L - (T_TRIP - self.m)) ** 2)
        w = 1.0 if obs["busy"] else d
        util = self.H * w * (self.wq * self.Qv + self.ws * np.minimum(tp, self.tgt) / self.tgt
                             - self.we * (Pa / tp) / self.E_ref)
        sw = np.array([self.mu if (self.last is not None and a != self.last) else 0.0
                       for a in self.acts])
        j = int(np.argmin(-util + pen + sw))
        self.last = self.acts[j]
        self._obs = dict(obs, Td=Tdm, Ts=Tsm)
        return self.acts[j][0], float(min(self.acts[j][1], cap)), self.acts[j][2]

    def feedback(self, obs_new, a, P_applied):
        if not self.use_bias: return
        Tdp, Tsp = thermal_step(self._obs["Td"], self._obs["Ts"], P_applied, self.p, self.T_amb)
        self.bd = float(np.clip(0.9 * self.bd + 0.1 * (obs_new["Td"] - Tdp), -2, 2))
        self.bs = float(np.clip(0.9 * self.bs + 0.1 * (obs_new["Ts"] - Tsp), -0.5, 0.5))

class ThermoRouteSpec(ThermoRoute):
    spec = True

def build(name, soc, Q, wl_stats, plant=None, q_table=None, **kw):
    if name == "EdgeMoE": return EdgeMoE(soc, Q)
    if name == "EdgeLLM": return EdgeLLM(soc, Q)
    if name == "ReactiveDVFS": return ReactiveDVFS(soc, Q)
    if name == "FiDRL": return FiDRL(soc, Q, table=q_table)
    if name == "StaticK4": return StaticK4(soc, Q)
    if name == "ThermoRoute": return ThermoRoute(soc, Q, wl_stats, **kw)
    if name == "ThermoRoute+Spec": return ThermoRouteSpec(soc, Q, wl_stats, **kw)
    if name == "Oracle":      # same MPC with true plant, true ambient, noise free sensing
        return ThermoRoute(soc, Q, wl_stats, nominal=plant.p, T_amb=plant.T_amb, bias=False, **kw)
    raise ValueError(name)

In [ ]:
%%writefile sim/runner.py
"""Episode simulation, FiDRL training, and the full experiment grid with checkpointing
(a Colab disconnect resumes where it stopped)."""
import os, time, itertools, pickle, hashlib, numpy as np, pandas as pd
from concurrent.futures import ProcessPoolExecutor
from config import *
from model_profile import load_quality
from plant import Plant, tps, power_active
from workload import Workload, workload_stats
from controllers import build, FiDRL

Q = load_quality()
LOGPPL = {k: np.log(v[1]) for k, v in Q.items()}
SAMPLE_DIR = os.path.join(RESULTS_DIR, "samples"); os.makedirs(SAMPLE_DIR, exist_ok=True)
RUNS_CSV = os.path.join(RESULTS_DIR, "all_runs.csv")
FIDRL_PKL = os.path.join(RESULTS_DIR, "fidrl_tables.pkl")

def episode(method, soc_key, amb, wl, seed, mismatch=MISMATCH_DEFAULT, kw=None,
            q_table=None, trace=False, agent=None):
    rng = np.random.default_rng(seed * 7919 + int(amb) * 31 + (0 if wl == "chat" else 1))
    soc = SOCS[soc_key]
    plant = Plant(soc, amb, np.random.default_rng(rng.integers(1 << 31)), mismatch)
    work = Workload(WORKLOADS[wl], np.random.default_rng(rng.integers(1 << 31)))
    ctrl = agent or build(method, soc, Q, workload_stats(WORKLOADS[wl]), plant=plant,
                          q_table=q_table, **(kw or {}))
    noisy = method != "Oracle"
    Td_m, Ts_m = plant.measure(noisy)
    rec = {x: [] for x in ("Td", "Ts", "k", "f", "tps", "a", "P", "cap")}
    E = tok = qtok = lptok = die_s = skin_s = anyg = 0.0
    k_time = {k: 0.0 for k in K_LEVELS}
    ovh, pk_d, pk_s = [], plant.Td, plant.Ts
    for t in range(SIM_SECONDS):
        busy = work.busy()
        obs = dict(Td=Td_m, Ts=Ts_m, cap=plant.cap(), busy=busy)
        t0 = time.perf_counter(); k, f, core = ctrl.act(obs); dt_ctrl = time.perf_counter() - t0
        if busy: ovh.append(dt_ctrl)                    # overhead on active decisions only
        fe = min(f, plant.cap())
        r = float(tps(soc, k, fe, core, ctrl.spec))
        a, dec = work.advance(r)
        P = a * float(power_active(soc, k, fe, core, ctrl.spec)) + (1 - a) * soc["idle_w"]
        die_s += plant.gov_die; skin_s += plant.gov_skin; anyg += (plant.gov_die or plant.gov_skin)
        cap_now = plant.cap()
        plant.step(P); E += P
        pk_d, pk_s = max(pk_d, plant.Td), max(pk_s, plant.Ts)
        tok += dec; qtok += dec * Q[k][0]; lptok += dec * LOGPPL[k]
        k_time[k] += a
        Td_m, Ts_m = plant.measure(noisy)
        obs_new = dict(Td=Td_m, Ts=Ts_m, cap=plant.cap(), busy=work.busy())
        ctrl.feedback(obs_new, a, P)
        if isinstance(ctrl, FiDRL) and ctrl.eps > 0:
            ctrl.learn(obs_new, dec / 10 - 5.0 * plant.gov_die
                       - 2.0 * max(0.0, plant.Ts - T_SKIN) - 0.1 * P)
        if trace:
            for x, v in zip(rec, (plant.Td, plant.Ts, k, fe, r * (a > 0), a, P, cap_now)):
                rec[x].append(v)
    lat = np.array([d[0] for d in work.done]); ttft = np.array([d[1] for d in work.done])
    dtps = np.array([d[2] for d in work.done])
    act = sum(k_time.values()) or 1.0
    nan = float("nan")
    res = dict(method=method, soc=soc_key, amb=amb, wl=wl, seed=seed, mismatch=mismatch,
               tokens=tok, qwt=qtok,
               eff_ppl=float(np.exp(lptok / tok)) if tok > 0 else nan,   # token weighted geometric
               peak_die=pk_d, peak_skin=pk_s,
               die_trip_s=die_s, skin_cap_s=skin_s, throttle_s=anyg, energy_J=E,
               j_per_tok=E / tok if tok else nan,
               tok_per_charge=tok / E * BATTERY_J if E else nan,
               responses=len(lat),
               lat_p50=np.percentile(lat, 50) if len(lat) else nan,
               lat_p95=np.percentile(lat, 95) if len(lat) else nan,
               ttft_p95=np.percentile(ttft, 95) if len(lat) else nan,
               slo_viol_pct=100 * np.mean(dtps < SLO_TPS) if len(lat) else nan,
               ovh_med_ms=1e3 * np.median(ovh) if ovh else nan,
               ovh_p99_ms=1e3 * np.percentile(ovh, 99) if ovh else nan,
               **{f"frac_k{k}": k_time[k] / act for k in K_LEVELS})
    for c in ("H", "skin_margin", "lam"):
        res[c] = (kw or {}).get(c, nan)
    return res, (rec if trace else None), dict(lat=lat, ovh=np.array(ovh) * 1e3)

def train_fidrl(soc_key, wl, schedule=(0.3, 0.15, 0.05)):
    """Episodes on training seeds at every ambient; exploration decays per pass.
    Plant, workload and controller state are rebuilt every episode."""
    agent = FiDRL(SOCS[soc_key], Q, eps=schedule[0], rng=np.random.default_rng(1))
    for eps in schedule:
        agent.eps = eps
        for s in SEEDS_TRAIN:
            for amb in AMBIENTS:
                agent.prev = None
                episode("FiDRL", soc_key, amb, wl, s, agent=agent)
    return agent.table

def get_fidrl_tables():
    if os.path.exists(FIDRL_PKL):
        with open(FIDRL_PKL, "rb") as f:
            return pickle.load(f)
    tables = {}
    for soc_key in SOCS:
        for wl in WORKLOADS:
            tables[(soc_key, wl)] = train_fidrl(soc_key, wl)
            print("FiDRL trained", soc_key, wl, flush=True)
    with open(FIDRL_PKL, "wb") as f:
        pickle.dump(tables, f)
    return tables

def _tune_one(args):
    w, (sk, amb, wl), s = args
    r, _, _ = episode("ThermoRoute", sk, amb, wl, s, kw=dict(w=w))
    return dict(**w, soc=sk, amb=amb, wl=wl, seed=s, qwt_per_J=r["qwt"] / r["energy_J"],
                throttle_s=r["throttle_s"], slo=r["slo_viol_pct"], lat_p95=r["lat_p95"])

def tune_weights(n_seeds=2):
    """Select MPC stage weights on TRAINING seeds only (never on test seeds).
    Rule: among settings with mean throttle <= 1 % of the episode and mean SLO violation
    <= 5 %, take the highest quality weighted tokens per joule."""
    import json
    if os.path.exists(MPC_WEIGHTS_JSON):
        best = json.load(open(MPC_WEIGHTS_JSON)); MPC.update(best)
        print("MPC weights (saved):", best); return best
    grid = [dict(zip(TUNE_GRID, v)) for v in itertools.product(*TUNE_GRID.values())]
    jobs = list(itertools.product(grid, TUNE_CONDS, SEEDS_TRAIN[:n_seeds]))
    with ProcessPoolExecutor(N_JOBS) as ex:
        df = pd.DataFrame(list(ex.map(_tune_one, jobs, chunksize=2)))
    df.to_csv(os.path.join(RESULTS_DIR, "mpc_tuning.csv"), index=False)
    g = df.groupby(list(TUNE_GRID))[["qwt_per_J", "throttle_s", "slo"]].mean().reset_index()
    ok = g[(g.throttle_s <= 0.01 * SIM_SECONDS) & (g.slo <= 5.0)]
    pick = (ok if len(ok) else g.sort_values("throttle_s").head(3)).sort_values(
        "qwt_per_J", ascending=False).iloc[0]
    best = {k: float(pick[k]) for k in TUNE_GRID}
    json.dump(best, open(MPC_WEIGHTS_JSON, "w")); MPC.update(best)
    print(g.round(3).to_string(index=False)); print("selected MPC weights:", best)
    return best

def job_list(tables):
    jobs = []
    def add(tag, m, sk, amb, wl, s, mismatch=MISMATCH_DEFAULT, kw=None, save=False):
        jid = f"{tag}|{m}|{sk}|{amb:g}|{wl}|{s}|{mismatch:g}|" + \
              ",".join(f"{a}={b:g}" for a, b in sorted((kw or {}).items()))
        jobs.append(dict(jid=jid, tag=tag, save=save, method=m, soc_key=sk, amb=amb, wl=wl,
                         seed=s, mismatch=mismatch, kw=kw,
                         q_table=tables[(sk, wl)] if m == "FiDRL" else None))
    for m, sk, amb, wl, s in itertools.product(METHODS, SOCS, AMBIENTS, WORKLOADS, SEEDS_TEST):
        add("main", m, sk, amb, wl, s, save=True)
    sc = STRESS_COND
    for H, mg, s in itertools.product(SWEEP_H, SWEEP_M, SEEDS_TEST):
        add("HM", "ThermoRoute", sc["soc"], sc["amb"], sc["wl"], s, kw=dict(H=H, skin_margin=mg))
    for lam, s in itertools.product(SWEEP_LAM, SEEDS_TEST):
        add("lam", "ThermoRoute", sc["soc"], sc["amb"], sc["wl"], s, kw=dict(lam=lam))
    for mm, m, s in itertools.product(SWEEP_MISMATCH, ["EdgeMoE", "ReactiveDVFS", "FiDRL",
                                                     "ThermoRoute"], SEEDS_TEST):
        add("mismatch", m, sc["soc"], sc["amb"], sc["wl"], s, mismatch=mm)
    return jobs

def run_job(job):
    job = dict(job)
    jid, tag, save = job.pop("jid"), job.pop("tag"), job.pop("save")
    res, rec, extra = episode(**job, trace=save)
    res["sweep"], res["jid"] = tag, jid
    tr = None
    if rec is not None:
        if job["seed"] == 0:
            tr = rec
        np.savez_compressed(os.path.join(SAMPLE_DIR, hashlib.md5(jid.encode()).hexdigest()[:16] + ".npz"),
                            jid=jid, lat=extra["lat"], ovh=extra["ovh"])
    return res, tr

def run_all(flush_every=40):
    tune_weights()
    tables = get_fidrl_tables()
    jobs = job_list(tables)
    done = set(pd.read_csv(RUNS_CSV, usecols=["jid"]).jid) if os.path.exists(RUNS_CSV) else set()
    todo = [j for j in jobs if j["jid"] not in done]
    print(f"jobs total {len(jobs)}, done {len(done)}, to run {len(todo)}, workers {N_JOBS}",
          flush=True)
    buf, t0 = [], time.time()
    def flush():
        if not buf: return
        df = pd.DataFrame(buf)
        if os.path.exists(RUNS_CSV):
            cols = pd.read_csv(RUNS_CSV, nrows=0).columns
            df.reindex(columns=cols).to_csv(RUNS_CSV, mode="a", header=False, index=False)
        else:
            df.to_csv(RUNS_CSV, index=False)
        buf.clear()
    with ProcessPoolExecutor(N_JOBS) as ex:
        for i, (res, rec) in enumerate(ex.map(run_job, todo, chunksize=2), 1):
            buf.append(res)
            if rec is not None:
                name = f"{res['method']}_{res['soc']}_{int(res['amb'])}_{res['wl']}.npz"
                np.savez_compressed(os.path.join(TRACE_DIR, name.replace("+", "p")),
                                    **{k: np.array(v) for k, v in rec.items()})
            if len(buf) >= flush_every:
                flush()
                print(f"{i}/{len(todo)}  {(time.time() - t0) / 60:.1f} min", flush=True)
    flush()
    print("all runs saved to", RUNS_CSV)

In [ ]:
%%writefile sim/stats_tables.py
"""Per condition mean and 95% t CI, paired Wilcoxon (ThermoRoute vs each method) with
direction and Holm correction, and IEEE LaTeX tables for MAIN_COND and STRESS_COND."""
import os, numpy as np, pandas as pd
from scipy import stats
from config import RESULTS_DIR, MAIN_COND, STRESS_COND, METHODS

METRICS = ["peak_die", "peak_skin", "throttle_s", "eff_ppl", "lat_p95", "ttft_p95",
           "slo_viol_pct", "j_per_tok", "tok_per_charge", "responses", "ovh_med_ms"]
HIGHER_BETTER = {"tok_per_charge", "responses"}
TEST = ["throttle_s", "peak_skin", "eff_ppl", "lat_p95", "slo_viol_pct", "j_per_tok",
        "tok_per_charge", "responses"]
TAB_COLS = [("peak_skin", r"Peak skin ($^\circ$C)", 2), ("throttle_s", "Throttle (s)", 1),
            ("eff_ppl", "PPL", 2), ("lat_p95", "p95 lat. (s)", 2), ("slo_viol_pct", "SLO viol. (\\%)", 1),
            ("j_per_tok", "J/token", 4), ("tok_per_charge", "Tokens/charge ($10^3$)", 1)]

def ci95(x):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    if len(x) < 2: return np.nan
    return stats.t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x))

def cond(d, c):
    return d[(d.soc == c["soc"]) & (d.amb == c["amb"]) & (d.wl == c["wl"])]

def holm(p):
    p = np.asarray(p, float); out = np.full_like(p, np.nan)
    ok = np.where(np.isfinite(p))[0]
    order = ok[np.argsort(p[ok])]; m = len(order); run = 0.0
    for r, i in enumerate(order):
        run = max(run, min(1.0, (m - r) * p[i])); out[i] = run
    return out

def wilcoxon_table(dc):
    rows = []
    tr = dc[dc.method == "ThermoRoute"].set_index("seed")
    for m in METHODS:
        if m == "ThermoRoute": continue
        o = dc[dc.method == m].set_index("seed")
        for k in TEST:
            a, b = tr[k].align(o[k], join="inner")
            diff = (a - b).to_numpy(); p = np.nan
            if len(diff) > 1 and np.any(diff != 0):
                p = stats.wilcoxon(a, b, zero_method="zsplit").pvalue
            md = float(np.median(diff))
            better = (md > 0) if k in HIGHER_BETTER else (md < 0)
            rows.append({"baseline": m, "metric": k, "thermoroute_mean": a.mean(),
                         "baseline_mean": b.mean(), "median_diff": md, "p": p,
                         "direction": "tie" if md == 0 else
                         ("ThermoRoute better" if better else "ThermoRoute worse")})
    w = pd.DataFrame(rows); w["p_holm"] = holm(w.p.to_numpy())
    w["significant_0.05"] = w.p_holm < 0.05
    return w

def latex_table(dc, caption, label):
    lines = [r"\begin{table}[!t]", r"\centering", r"\caption{" + caption + "}",
             r"\label{" + label + "}", r"\resizebox{\columnwidth}{!}{%",
             r"\begin{tabular}{l" + "c" * len(TAB_COLS) + "}", r"\hline",
             "Method & " + " & ".join(h for _, h, _ in TAB_COLS) + r" \\", r"\hline"]
    for m in METHODS:
        r = dc[dc.method == m]; cells = []
        for k, _, dp in TAB_COLS:
            sc = 1e-3 if k == "tok_per_charge" else 1.0
            cells.append(f"${r[k].mean() * sc:.{dp}f}\\pm{ci95(r[k]) * sc:.{dp}f}$")
        name = r"\textbf{" + m + "}" if m == "ThermoRoute" else m
        lines.append(name + " & " + " & ".join(cells) + r" \\")
    lines += [r"\hline", r"\end{tabular}}", r"\end{table}"]
    return "\n".join(lines)

def main():
    d = pd.read_csv(os.path.join(RESULTS_DIR, "all_runs.csv"))
    mr = d[d.sweep == "main"]
    g = mr.groupby(["soc", "amb", "wl", "method"])
    summ = g[METRICS].mean().add_suffix("_mean").join(g[METRICS].agg(ci95).add_suffix("_ci"))
    summ.reset_index().to_csv(os.path.join(RESULTS_DIR, "summary_all_conditions.csv"), index=False)
    for tag, c in (("main", MAIN_COND), ("stress", STRESS_COND)):
        dc = cond(mr, c)
        wilcoxon_table(dc).to_csv(os.path.join(RESULTS_DIR, f"wilcoxon_{tag}.csv"), index=False)
        cap = (f"Results at {c['soc']}, {c['amb']:g} \\textdegree C ambient, {c['wl']} workload "
               f"(mean $\\pm$ 95\\% CI over {dc.seed.nunique()} seeds)")
        with open(os.path.join(RESULTS_DIR, f"table_{tag}.tex"), "w") as f:
            f.write(latex_table(dc, cap, f"tab:{tag}"))
    print(open(os.path.join(RESULTS_DIR, "table_main.tex")).read())

if __name__ == "__main__":
    main()

In [ ]:
%%writefile sim/figs.py
"""All result figures (IEEE style: Times New Roman, 8 pt, 800 dpi, single 3.5 in / double
7.16 in). Every number is read from results/ so figures always agree with the tables.
Panel tags sit below the x label; one shared legend sits under each figure."""
import os, glob
import numpy as np, pandas as pd
import matplotlib as mpl
mpl.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
from scipy import stats
from config import (RESULTS_DIR, TRACE_DIR, DATA_DIR, FIG_DIR, MAIN_COND, STRESS_COND, METHODS,
                    K_LEVELS, T_TRIP, T_SKIN, AMBIENTS, SOCS, SWEEP_H, SWEEP_M)

for _f in fm.findSystemFonts():
    if any(s in os.path.basename(_f).lower() for s in ("times", "liberationserif")):
        try: fm.fontManager.addfont(_f)
        except Exception: pass

SINGLE, DOUBLE, DPI = 3.5, 7.16, 800
mpl.rcParams.update({
    "font.family": "serif", "font.serif": ["Times New Roman", "Liberation Serif", "DejaVu Serif"],
    "mathtext.fontset": "stix", "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8,
    "legend.fontsize": 7, "xtick.labelsize": 7, "ytick.labelsize": 7,
    "axes.linewidth": 0.6, "lines.linewidth": 1.0, "lines.markersize": 3.5,
    "xtick.direction": "in", "ytick.direction": "in", "xtick.major.width": 0.6,
    "ytick.major.width": 0.6, "legend.frameon": False, "axes.grid": True,
    "grid.linewidth": 0.3, "grid.alpha": 0.4, "savefig.dpi": DPI,
    "savefig.bbox": "tight", "savefig.pad_inches": 0.02, "pdf.fonttype": 42, "ps.fonttype": 42})

STYLE = {  # color, marker, linestyle, hatch: readable in grayscale print
    "EdgeMoE":          ("#1f77b4", "o", "-", ""),
    "EdgeLLM":          ("#ff7f0e", "s", "--", "//"),
    "ReactiveDVFS":     ("#8c564b", "v", "-.", "\\\\"),
    "FiDRL":            ("#2ca02c", "^", ":", "xx"),
    "StaticK4":         ("#9467bd", "D", (0, (3, 1, 1, 1)), ".."),
    "ThermoRoute":      ("#d62728", "*", "-", ""),
    "ThermoRoute+Spec": ("#e377c2", "P", "--", "--"),
    "Oracle":           ("#7f7f7f", "x", ":", "++"),
}
LABEL = {"peak_skin": "Peak skin temp. (°C)", "throttle_s": "Throttle time (s)",
         "eff_ppl": "Effective perplexity", "j_per_tok": "Energy per token (J)",
         "lat_p95": "p95 latency (s)", "tok_per_charge": "Tokens per charge"}

def save(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name + ".png"))
    fig.savefig(os.path.join(FIG_DIR, name + ".pdf"))
    plt.close(fig); print("saved", name)

def legend_below(fig, handles, labels, ncol=None):
    fig.legend(handles, labels, ncol=ncol or min(len(labels), 4), loc="upper center",
               bbox_to_anchor=(0.5, 0.0), handlelength=2.2, columnspacing=1.0)

def tag(ax, letter, xlabel):
    ax.set_xlabel(f"{xlabel}\n({letter})")

def runs(sweep="main"):
    d = pd.read_csv(os.path.join(RESULTS_DIR, "all_runs.csv"))
    return d[d.sweep == sweep]

def cond(d, c):
    return d[(d.soc == c["soc"]) & (d.amb == c["amb"]) & (d.wl == c["wl"])]

def ci95(x):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    return stats.t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else 0.0

def trace(method, c):
    name = f"{method}_{c['soc']}_{int(c['amb'])}_{c['wl']}.npz".replace("+", "p")
    return np.load(os.path.join(TRACE_DIR, name))

def cname(c):
    return f"{c['soc']}, {c['amb']:g} °C, {c['wl']}"

# ---------------------------------------------------------------------------------------------
def fig_A():
    """Measured quality, analytic phone speed, measured expert union (double column)."""
    from plant import tps
    kp = pd.read_csv(os.path.join(DATA_DIR, "k_profile.csv"))
    un = pd.read_csv(os.path.join(DATA_DIR, "expert_union.csv"))
    p = SOCS[MAIN_COND["soc"]]
    fig, ax = plt.subplots(1, 3, figsize=(DOUBLE, 1.75))
    ax[0].errorbar(kp.k, kp.ppl, yerr=kp.get("ppl_ci", 0 * kp.ppl), fmt="o-", color="#d62728",
                   capsize=2, elinewidth=0.6)
    ax[0].set_ylabel("WikiText 103 perplexity"); ax[0].set_xticks(K_LEVELS)
    tag(ax[0], "a", "Active experts $k$")
    h1, = ax[1].plot(K_LEVELS, [float(tps(p, k, 1.0, "performance")) for k in K_LEVELS], "o-",
                     color="#1f77b4", label="Plain decoding")
    h2, = ax[1].plot(K_LEVELS, [float(tps(p, k, 1.0, "performance", True)) for k in K_LEVELS],
                     "s--", color="#ff7f0e", label="Speculative decoding")
    ax[1].set_ylabel("Phone decode speed (tok/s)"); ax[1].set_xticks(K_LEVELS)
    tag(ax[1], "b", "Active experts $k$")
    for i, k in enumerate(K_LEVELS):
        u = un[un.k == k].sort_values("window")
        ax[2].plot(u.window, u.union, marker="os^D"[i], ls="-", color=plt.cm.Greys(0.35 + 0.18 * i),
                   label=f"$k={k}$")
    ax[2].set_ylabel("Distinct experts per layer"); ax[2].set_xticks(sorted(un.window.unique()))
    tag(ax[2], "c", "Consecutive tokens in window")
    fig.tight_layout(w_pad=1.0)
    h3, l3 = ax[2].get_legend_handles_labels()
    legend_below(fig, [h1, h2] + h3, [h1.get_label(), h2.get_label()] + l3, ncol=6)
    save(fig, "fig_A_k_profile")

def fig_B():
    """Temporal dynamics, STRESS_COND, seed 0 (double column, stacked panels)."""
    c = STRESS_COND
    show = ["EdgeMoE", "ReactiveDVFS", "FiDRL", "StaticK4", "ThermoRoute"]
    fig, ax = plt.subplots(5, 1, figsize=(DOUBLE, 5.0), sharex=True,
                           gridspec_kw=dict(height_ratios=[1, 1, 0.75, 1, 0.75]))
    for m in show:
        tr = trace(m, c); t = np.arange(len(tr["Td"])); col, _, ls, _ = STYLE[m]
        ax[0].plot(t, tr["Td"], color=col, ls=ls, lw=0.9, label=m)
        ax[1].plot(t, tr["Ts"], color=col, ls=ls, lw=0.9)
        if m == "ThermoRoute":                            # k only matters while decoding
            kk = pd.Series(np.where(tr["a"] > 0, tr["k"], np.nan)).rolling(
                30, center=True, min_periods=1).mean()
            ax[2].plot(t, kk, color=col, ls=ls, lw=0.9)
        ax[3].plot(t, pd.Series(tr["tps"]).rolling(15, center=True, min_periods=1).mean(),
                   color=col, ls=ls, lw=0.9)
        ax[4].step(t, tr["cap"], color=col, ls=ls, lw=0.8, where="post")
    ax[0].axhline(T_TRIP, color="k", ls="--", lw=0.6)
    ax[1].axhline(T_SKIN, color="k", ls="--", lw=0.6)
    ax[0].set_ylabel("Die temp. (°C)"); ax[1].set_ylabel("Skin temp. (°C)")
    ax[2].set_ylabel("ThermoRoute $k$\n(30 s mean)"); ax[2].set_yticks(K_LEVELS)
    ax[2].set_ylim(min(K_LEVELS) - 0.5, max(K_LEVELS) + 0.5)
    ax[3].set_ylabel("Tok/s (15 s avg.)"); ax[4].set_ylabel("Governor cap")
    ax[4].set_xlabel("Time (s)")
    for a, l in zip(ax, "abcde"):
        a.text(1.005, 0.5, f"({l})", transform=a.transAxes, va="center", ha="left")
    fig.align_ylabels(ax); fig.tight_layout(h_pad=0.3)
    h, l = ax[0].get_legend_handles_labels(); legend_below(fig, h, l, ncol=len(show))
    save(fig, "fig_B_dynamics")

def fig_C():
    """Energy per token versus effective perplexity, mean and 95% CI (double column)."""
    d = runs()
    fig, ax = plt.subplots(1, 2, figsize=(DOUBLE, 2.3))
    for a, c, l in zip(ax, (MAIN_COND, STRESS_COND), "ab"):
        dc = cond(d, c)
        for m in METHODS:
            r = dc[dc.method == m]; col, mk, _, _ = STYLE[m]
            a.errorbar(r.eff_ppl.mean(), r.j_per_tok.mean(), xerr=ci95(r.eff_ppl),
                       yerr=ci95(r.j_per_tok), fmt=mk, color=col, ms=5, capsize=2,
                       elinewidth=0.6, label=m)
        a.set_ylabel(LABEL["j_per_tok"]); tag(a, l, f"{LABEL['eff_ppl']}  [{cname(c)}]")
    fig.tight_layout(w_pad=1.2)
    h, lab = ax[0].get_legend_handles_labels(); legend_below(fig, h, lab)
    save(fig, "fig_C_pareto")

def _samples(c, kind):
    out = {}
    for f in glob.glob(os.path.join(RESULTS_DIR, "samples", "*.npz")):
        z = np.load(f, allow_pickle=True); jid = str(z["jid"]).split("|")
        if jid[0] != "main" or jid[2] != c["soc"] or float(jid[3]) != c["amb"] or jid[4] != c["wl"]:
            continue
        out.setdefault(jid[1], []).append(z[kind])
    return {m: np.concatenate(v) for m, v in out.items()}

def fig_D():
    """Response latency CDF (double column)."""
    fig, ax = plt.subplots(1, 2, figsize=(DOUBLE, 2.2))
    for a, c, l in zip(ax, (MAIN_COND, STRESS_COND), "ab"):
        s = _samples(c, "lat")
        for m in METHODS:
            x = np.sort(s[m]); col, _, ls, _ = STYLE[m]
            a.plot(x, np.arange(1, len(x) + 1) / len(x), color=col, ls=ls, lw=1.0, label=m)
        a.set_xscale("log"); a.set_ylabel("CDF"); a.set_ylim(0, 1.0)
        tag(a, l, f"Response latency (s)  [{cname(c)}]")
    fig.tight_layout(w_pad=1.2)
    h, lab = ax[0].get_legend_handles_labels(); legend_below(fig, h, lab)
    save(fig, "fig_D_latency_cdf")

def _heat(a, M, xt, yt, fmt, cmap):
    im = a.imshow(M, cmap=cmap, aspect="auto"); a.grid(False)
    a.set_xticks(range(len(xt))); a.set_xticklabels(xt); a.set_yticks(range(len(yt)))
    a.set_yticklabels(yt)
    lo, hi = np.nanmin(M), np.nanmax(M)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M[i, j]; dark = hi > lo and (v - lo) / (hi - lo) > 0.6
            a.text(j, i, fmt(v), ha="center", va="center", fontsize=7,
                   color="w" if dark else "k")
    return im

def fig_E():
    """Horizon x margin sensitivity on STRESS_COND, raw values (double column)."""
    d = runs("HM"); g = d.groupby(["H", "skin_margin"])
    fig, ax = plt.subplots(1, 3, figsize=(DOUBLE, 1.9))
    for a, (k, lab), l in zip(ax, [("throttle_s", "Throttle time (s)"),
                                   ("peak_skin", "Peak skin temp. (°C)"),
                                   ("eff_ppl", "Effective perplexity")], "abc"):
        M = g[k].mean().unstack("skin_margin").reindex(index=SWEEP_H, columns=SWEEP_M).to_numpy()
        im = _heat(a, M, [f"{m:g}" for m in SWEEP_M], [f"{h:g}" for h in SWEEP_H],
                   (lambda v: f"{v:.1f}") if k != "eff_ppl" else (lambda v: f"{v:.2f}"), "Greys")
        a.set_ylabel("Horizon $H$ (s)"); tag(a, l, f"Skin margin (°C): {lab}")
        fig.colorbar(im, ax=a, fraction=0.05, pad=0.03)
    fig.tight_layout(w_pad=0.8); save(fig, "fig_E_sensitivity")

def fig_F():
    """Model mismatch robustness on STRESS_COND (double column)."""
    d = runs("mismatch"); show = ["EdgeMoE", "ReactiveDVFS", "FiDRL", "ThermoRoute"]
    fig, ax = plt.subplots(1, 3, figsize=(DOUBLE, 1.9))
    for a, k, l in zip(ax, ["peak_skin", "throttle_s", "eff_ppl"], "abc"):
        for m in show:
            g = d[d.method == m].groupby("mismatch")[k]; col, mk, ls, _ = STYLE[m]
            a.errorbar(g.mean().index * 100, g.mean(), yerr=g.agg(ci95), color=col, marker=mk,
                       ls=ls, capsize=2, elinewidth=0.6, label=m)
        a.set_ylabel(LABEL[k]); tag(a, l, "Plant R/C mismatch (± %)")
    fig.tight_layout(w_pad=1.0)
    h, lab = ax[0].get_legend_handles_labels(); legend_below(fig, h, lab)
    save(fig, "fig_F_mismatch")

def fig_G():
    """Ambient stress for both SoCs, agent workload (double column, 2 x 3 bars)."""
    d = runs(); d = d[d.wl == "agent"]
    ref = d[d.method == "EdgeMoE"].groupby(["soc", "amb"]).eff_ppl.mean()
    fig, ax = plt.subplots(2, 3, figsize=(DOUBLE, 3.6), sharex=True)
    w = 0.8 / len(METHODS); x = np.arange(len(AMBIENTS))
    for i, soc in enumerate(SOCS):
        for j, (k, lab) in enumerate([("throttle_s", "Throttle time (s)"),
                                      ("ppl_inc", "Perplexity increase (%)"),
                                      ("j_per_tok", "Energy per token (J)")]):
            a = ax[i, j]
            for n, m in enumerate(METHODS):
                r = d[(d.soc == soc) & (d.method == m)].copy()
                if k == "ppl_inc":
                    r[k] = 100 * (r.eff_ppl / r.amb.map(lambda t: ref[(soc, t)]) - 1)
                g = r.groupby("amb")[k]; col, _, _, hat = STYLE[m]
                a.bar(x + (n - (len(METHODS) - 1) / 2) * w, g.mean().reindex(AMBIENTS), w,
                      yerr=g.agg(ci95).reindex(AMBIENTS), color=col, hatch=hat, ec="k", lw=0.3,
                      error_kw=dict(lw=0.5, capsize=1), label=m)
            a.set_ylabel(lab); a.set_ylim(bottom=0); a.grid(axis="x", visible=False)
            a.set_xticks(x); a.set_xticklabels([f"{t:g} °C" for t in AMBIENTS])
            if i == 1: tag(a, "abcdef"[i * 3 + j], f"Ambient, {soc}")
            else: a.set_xlabel(f"({'abcdef'[j]}) {soc}")
    fig.tight_layout(h_pad=0.8, w_pad=0.8)
    h, lab = ax[0, 0].get_legend_handles_labels(); legend_below(fig, h, lab)
    save(fig, "fig_G_ambient")

def fig_H():
    """Share of active time per expert budget, SoC-A agent, mean over seeds (single column)."""
    d = runs(); d = d[(d.soc == MAIN_COND["soc"]) & (d.wl == "agent")]
    show = ["ThermoRoute", "ThermoRoute+Spec", "Oracle"]
    cols = [f"frac_k{k}" for k in K_LEVELS]
    shades = plt.cm.Greys(np.linspace(0.15, 0.85, len(K_LEVELS)))
    fig, ax = plt.subplots(figsize=(SINGLE, 2.1))
    pos, labels, p = [], [], 0.0
    for m in show:
        for amb in AMBIENTS:
            fr = d[(d.method == m) & (d.amb == amb)][cols].mean().to_numpy()
            bottom = 0.0
            for j, v in enumerate(fr):
                ax.bar(p, v, 0.8, bottom=bottom, color=shades[j], ec="k", lw=0.3,
                       label=f"$k={K_LEVELS[j]}$" if p == 0 else None)
                bottom += v
            pos.append(p); labels.append(f"{amb:g}"); p += 1
        p += 0.6
    ax.set_xticks(pos); ax.set_xticklabels(labels)
    for n, m in enumerate(show):
        ax.text(np.mean(pos[n * 3:(n + 1) * 3]), -0.11, m, ha="center", va="top", fontsize=7,
                transform=ax.get_xaxis_transform())
    ax.set_ylabel("Fraction of active time"); ax.set_ylim(0, 1); ax.grid(axis="x", visible=False)
    ax.set_xlabel("Ambient (°C)", labelpad=11)
    fig.tight_layout()
    h, lab = ax.get_legend_handles_labels(); legend_below(fig, h, lab)
    save(fig, "fig_H_k_occupancy")

def fig_I():
    """Per decision controller overhead on active decisions (single column)."""
    s = _samples(MAIN_COND, "ovh")
    data = [np.maximum(s[m], 1e-4) for m in METHODS]
    fig, ax = plt.subplots(figsize=(SINGLE, 2.0))
    bp = ax.boxplot(data, patch_artist=True, showfliers=False, widths=0.6,
                    medianprops=dict(color="k", lw=0.8), whiskerprops=dict(lw=0.6),
                    capprops=dict(lw=0.6), boxprops=dict(lw=0.6))
    for b, m in zip(bp["boxes"], METHODS):
        b.set_facecolor(STYLE[m][0]); b.set_hatch(STYLE[m][3]); b.set_alpha(0.8)
    ax.set_yscale("log"); ax.set_ylabel("Decision time (ms)")
    ax.set_xticks(range(1, len(METHODS) + 1))
    ax.set_xticklabels(METHODS, rotation=35, ha="right"); ax.grid(axis="x", visible=False)
    fig.tight_layout(); save(fig, "fig_I_overhead")

def fig_J():
    """Method x metric table on STRESS_COND: raw means annotated, shade = rank in column."""
    d = cond(runs(), STRESS_COND)
    cols = [("peak_skin", "Peak skin", -1), ("throttle_s", "Throttle (s)", -1),
            ("eff_ppl", "PPL", -1), ("lat_p95", "p95 lat. (s)", -1),
            ("slo_viol_pct", "SLO viol. (%)", -1), ("j_per_tok", "J/token", -1),
            ("tok_per_charge", "Tok/charge", 1), ("responses", "Responses", 1)]
    raw = d.groupby("method")[[c for c, _, _ in cols]].mean().reindex(METHODS)
    rank = raw.copy()
    for c, _, s in cols:
        rank[c] = raw[c].rank(ascending=(s < 0), method="average")
    fig, ax = plt.subplots(figsize=(DOUBLE, 2.5))
    ax.imshow(rank.to_numpy(), cmap="Greys", vmin=0.5, vmax=len(METHODS) + 3, aspect="auto")
    ax.set_xticks(range(len(cols))); ax.set_xticklabels([l for _, l, _ in cols])
    ax.set_yticks(range(len(METHODS))); ax.set_yticklabels(METHODS); ax.grid(False)
    ax.xaxis.tick_top()
    for i in range(len(METHODS)):
        for j, (c, _, _) in enumerate(cols):
            v = raw.iat[i, j]
            txt = ("0" if v == 0 else f"{v / 1000:.0f}k" if abs(v) >= 10000 else
                   f"{v:.0f}" if abs(v) >= 100 else f"{v:.1f}" if abs(v) >= 1 else f"{v:.3f}")
            ax.text(j, i, txt, ha="center", va="center", fontsize=7)
    ax.set_xlabel(f"Shade: rank in column, lighter is better  [{cname(STRESS_COND)}]")
    fig.tight_layout(); save(fig, "fig_J_normalized")

def fig_K():
    """Thermal penalty weight trade off on STRESS_COND (double column)."""
    d = runs("lam"); g = d.groupby("lam")
    fig, ax = plt.subplots(1, 3, figsize=(DOUBLE, 1.9))
    for a, k, l in zip(ax, ["throttle_s", "eff_ppl", "j_per_tok"], "abc"):
        a.errorbar(g[k].mean().index, g[k].mean(), yerr=g[k].agg(ci95), marker="*",
                   color=STYLE["ThermoRoute"][0], capsize=2, lw=0.9, elinewidth=0.5)
        a.set_xscale("log"); a.set_ylabel(LABEL[k]); tag(a, l, r"Penalty weight $\lambda$")
    fig.tight_layout(w_pad=1.0); save(fig, "fig_K_lambda")

def make_all():
    for f in (fig_A, fig_B, fig_C, fig_D, fig_E, fig_F, fig_G, fig_H, fig_I, fig_J, fig_K):
        f()
    print("figures in", FIG_DIR)

## Step 1: measured OLMoE profile (GPU)
Loads OLMoE 1B 7B in 8 bit, overrides the router top k in every MoE block, and measures:
1. WikiText 103 test perplexity for each k (60 chunks of 1024 tokens) with a bootstrap 95% CI.
2. Distinct experts per layer over windows of 1 to 5 consecutive tokens, read from router logits
   (this sets the true cost of speculative verification for an MoE model).
3. GPU decode speed (informative only; phone speed is derived in the simulator).

The cell stops with an error if changing k does not change perplexity, so a silent failure of the
top k override cannot reach the paper.

In [ ]:
import json, time, numpy as np, pandas as pd
DATA = os.path.join(ROOT, "data"); os.makedirs(DATA, exist_ok=True)
HF_MODEL, K_LEVELS, GAMMA = "allenai/OLMoE-1B-7B-0924", [2, 4, 6, 8], 4
CTX, N_CHUNKS, N_UNION = 1024, 60, 12
F_CFG, F_KP, F_UN = [os.path.join(DATA, f) for f in ("olmoe_config.json", "k_profile.csv", "expert_union.csv")]

def union_ok():
    if not os.path.exists(F_UN): return False
    u = pd.read_csv(F_UN)
    return (len(u) == len(K_LEVELS) * (GAMMA + 1) and u.union.notna().all()
            and np.allclose(u[u.window == 1].sort_values("k").union, K_LEVELS))

need_ppl = not (os.path.exists(F_CFG) and os.path.exists(F_KP))
need_union = not union_ok()
if not need_ppl and not need_union:
    print("Step 1 already done:"); print(pd.read_csv(F_KP))
    print(pd.read_csv(F_UN).pivot(index="k", columns="window", values="union").round(2))
else:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from datasets import load_dataset
    assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU, then run again"
    tok = AutoTokenizer.from_pretrained(HF_MODEL)
    model = AutoModelForCausalLM.from_pretrained(
        HF_MODEL, device_map="cuda:0", torch_dtype=torch.float16,
        quantization_config=BitsAndBytesConfig(load_in_8bit=True))
    model.eval()
    cfg = model.config.to_dict(); E = cfg["num_experts"]
    blocks = [m for m in model.modules() if hasattr(m, "top_k") and hasattr(m, "experts")]
    assert len(blocks) == cfg["num_hidden_layers"], f"found {len(blocks)} MoE blocks"
    def set_k(k):
        for b in blocks: b.top_k = k
        model.config.num_experts_per_tok = k
    text = "\n\n".join(load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="test")["text"])
    ids = tok(text)["input_ids"]
    chunks = [ids[i * CTX:(i + 1) * CTX] for i in range(N_CHUNKS)]
    print(f"{len(ids)} WikiText tokens, using {N_CHUNKS * CTX}")

    if need_ppl:     # (a) perplexity and GPU decode speed per k
        rows, rng = [], np.random.default_rng(0)
        prompt = tok("The history of mobile computing", return_tensors="pt").to("cuda:0")
        model.generate(**prompt, max_new_tokens=8, do_sample=False)            # warm up
        for k in K_LEVELS:
            set_k(k); nll = []
            with torch.no_grad():
                for ch in chunks:
                    x = torch.tensor([ch], device="cuda:0")
                    nll.append(model(x, labels=x).loss.item())
            nll = np.array(nll)
            boot = np.exp([nll[rng.integers(0, len(nll), len(nll))].mean() for _ in range(2000)])
            torch.cuda.synchronize(); t0 = time.perf_counter()
            g = model.generate(**prompt, max_new_tokens=64, min_new_tokens=64, do_sample=False)
            torch.cuda.synchronize()
            rows.append(dict(k=k, ppl=float(np.exp(nll.mean())),
                             ppl_ci=float((np.percentile(boot, 97.5) - np.percentile(boot, 2.5)) / 2),
                             tps_gpu=(g.shape[1] - prompt["input_ids"].shape[1]) / (time.perf_counter() - t0),
                             n_tokens=N_CHUNKS * (CTX - 1)))
            print(rows[-1])
        kp = pd.DataFrame(rows)
        if not kp.ppl.iloc[0] > 1.01 * kp.ppl.iloc[-1]:
            raise RuntimeError("Perplexity did not change with k: the top k override failed. Do not continue.")
        json.dump(cfg, open(F_CFG, "w"), indent=1, default=str); kp.to_csv(F_KP, index=False)
        print(kp)

    if need_union:   # (b) distinct experts per layer over windows of consecutive tokens
        store = []
        hooks = [b.gate.register_forward_hook(lambda m, i, o: store.append(o.detach().float().cpu()))
                 for b in blocks]
        urows = []
        for k in K_LEVELS:
            set_k(k); store.clear()
            with torch.no_grad():
                for ch in chunks[:N_UNION]:
                    model(torch.tensor([ch], device="cuda:0"))
            assert len(store) == N_UNION * len(blocks), f"captured {len(store)} router outputs"
            for w in range(1, GAMMA + 2):
                vals = []
                for L in store:
                    oh = torch.zeros(L.shape[0], E, dtype=torch.bool)
                    oh.scatter_(1, L.topk(k, dim=-1).indices, True)
                    T = (L.shape[0] // w) * w
                    vals.append(oh[:T].view(-1, w, E).any(1).sum(-1).float().mean().item())
                urows.append(dict(k=k, window=w, union=float(np.mean(vals))))
        for h in hooks: h.remove()
        pd.DataFrame(urows).to_csv(F_UN, index=False)
        assert union_ok(), "expert union check failed (window 1 must equal k)"
        print(pd.DataFrame(urows).pivot(index="k", columns="window", values="union").round(2))
    del model; torch.cuda.empty_cache()

## Step 2: real LMSYS Chat 1M request lengths
Streams the dataset (no full download), shuffles with a fixed seed, keeps English conversations,
and tokenizes every user and assistant turn with the OLMoE tokenizer. Pairs longer than the
4096 token OLMoE context are dropped. Responses come from the models in the LMSYS logs, so they
are a length distribution for real chat, not OLMoE outputs (state this in the paper).

In [ ]:
import pandas as pd
N_CONV = 5000
REQ = os.path.join(ROOT, "data", "requests.csv")
if os.path.exists(REQ):
    print("Step 2 already done:"); print(pd.read_csv(REQ).describe().round(1))
else:
    from datasets import load_dataset
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained("allenai/OLMoE-1B-7B-0924")
    n_tok = lambda s: len(tok.encode(s or "", add_special_tokens=False))
    ds = load_dataset("lmsys/lmsys-chat-1m", split="train", streaming=True).shuffle(seed=0, buffer_size=20000)
    rows, n = [], 0
    for ex in ds:
        if ex.get("language") != "English": continue
        conv = ex["conversation"]
        for a, b in zip(conv[:-1], conv[1:]):
            if a["role"] == "user" and b["role"] == "assistant":
                pt, rt = n_tok(a["content"]), n_tok(b["content"])
                if rt > 0 and pt + rt <= 4096:
                    rows.append(dict(conversation_id=ex["conversation_id"], model=ex["model"],
                                     prompt_tokens=max(pt, 1), response_tokens=rt))
        n += 1
        if n % 500 == 0: print(n, "conversations", flush=True)
        if n >= N_CONV: break
    df = pd.DataFrame(rows); df.to_csv(REQ, index=False)
    print(len(df), "request pairs from", n, "conversations"); print(df.describe().round(1))

## Step 3: sanity checks (read these numbers before running the grid)

In [ ]:
import importlib, numpy as np, pandas as pd
import config; importlib.reload(config)
from config import SOCS, K_LEVELS, F_LEVELS, T_SKIN, WORKLOADS
import model_profile, plant, workload
for m in (model_profile, plant, workload): importlib.reload(m)
from plant import tps, power_active
rows = []
for s, p in SOCS.items():
    for k in K_LEVELS:
        rows.append(dict(soc=s, k=k, tps_plain=float(tps(p, k, 1.0, "performance")),
                         tps_spec=float(tps(p, k, 1.0, "performance", True)),
                         tps_eff_core=float(tps(p, k, 1.0, "efficiency")),
                         P_full_W=float(power_active(p, k, 1.0, "performance")),
                         P_min_W=float(power_active(p, k, min(F_LEVELS), "performance"))))
print(pd.DataFrame(rows).round(2).to_string(index=False))
print("\nSteady skin temperature (C) at 100% duty, f = 1 and f = min, and at idle:")
for s, p in SOCS.items():
    for amb in config.AMBIENTS:
        hi = amb + float(power_active(p, 8, 1.0, "performance")) * p["Rsa"]
        lo = amb + float(power_active(p, 8, min(F_LEVELS), "performance")) * p["Rsa"]
        print(f"  {s} {amb:g} C: full {hi:.1f}, min freq {lo:.1f}, idle {amb + p['idle_w'] * p['Rsa']:.1f}  (limit {T_SKIN})")
for wl, r in WORKLOADS.items():
    print(wl, "workload stats (mean prompt, response, think):", np.round(workload.workload_stats(r), 1))
print("quality Q(k):", {k: round(v[0], 4) for k, v in model_profile.load_quality().items()})

## Step 4: experiments
1. MPC stage weights are selected on **training seeds** only (rule printed below).
2. FiDRL is trained on training seeds at every ambient with decaying exploration.
3. The grid: 8 methods, 2 SoCs, 3 ambients, 2 workloads, 20 test seeds, plus horizon x skin margin,
   penalty weight and mismatch sweeps on the stress condition (SoC-A, 35 C, agent).

Results are appended to `results/all_runs.csv`; re running this cell resumes. To start fresh,
delete the `results` folder. For a 3 minute smoke test only, run
`os.environ["THERMOROUTE_QUICK"] = "1"` first and use a different TR_ROOT (never for the paper).

In [ ]:
import importlib, config, model_profile, plant, workload, controllers, runner
for m in (config, model_profile, plant, workload, controllers, runner): importlib.reload(m)
runner.run_all()

## Step 5: tables and statistics

In [ ]:
import importlib, stats_tables, pandas as pd; importlib.reload(stats_tables)
stats_tables.main()
pd.set_option("display.width", 200)
w = pd.read_csv(os.path.join(ROOT, "results", "wilcoxon_stress.csv"))
display(w[["baseline", "metric", "thermoroute_mean", "baseline_mean", "p_holm", "direction"]].round(4))

## Step 6: figures (PNG 800 dpi and PDF, IEEE sizes)

In [ ]:
import importlib, figs; importlib.reload(figs)
figs.make_all()
from IPython.display import Image, display
for n in ["fig_B_dynamics", "fig_G_ambient"]:
    display(Image(os.path.join(os.environ["TR_FIG_DIR"], n + ".png"), width=900))

## Step 7: claim check against the abstract, then download
Every abstract claim is checked against the Holm corrected Wilcoxon results. Rewrite any claim
marked NOT SUPPORTED before submission.

In [ ]:
import pandas as pd
R = os.path.join(ROOT, "results")
def verdict(tag, baselines, metric):
    w = pd.read_csv(os.path.join(R, f"wilcoxon_{tag}.csv"))
    w = w[(w.metric == metric) & (w.baseline.isin(baselines))]
    ok = (w.direction == "ThermoRoute better") & (w.p_holm < 0.05)
    return "SUPPORTED" if ok.all() else "NOT SUPPORTED (" + ", ".join(w.baseline[~ok]) + ")"
base = ["EdgeMoE", "EdgeLLM", "StaticK4", "ReactiveDVFS", "FiDRL"]
for tag in ("main", "stress"):
    print(f"--- {tag} condition ---")
    print("smoother throughput (fewer throttle seconds):", verdict(tag, base, "throttle_s"))
    print("lower peak temperature (skin):              ", verdict(tag, base, "peak_skin"))
    print("higher tokens per charge:                   ", verdict(tag, base, "tok_per_charge"))
    print("lower p95 latency:                          ", verdict(tag, base, "lat_p95"))
    print("quality cost (PPL) vs each baseline:")
    w = pd.read_csv(os.path.join(R, f"wilcoxon_{tag}.csv")); w = w[w.metric == "eff_ppl"]
    print(w[["baseline", "thermoroute_mean", "baseline_mean", "direction", "p_holm"]].round(3).to_string(index=False))

In [ ]:
import shutil
pack = os.path.join(ROOT, "ThermoRoute_pack"); os.makedirs(pack, exist_ok=True)
for sub in ("data", "results"):
    shutil.copytree(os.path.join(ROOT, sub), os.path.join(pack, sub), dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns("samples", "traces"))
shutil.copytree(os.environ["TR_FIG_DIR"], os.path.join(pack, "figures"), dirs_exist_ok=True)
z = shutil.make_archive(os.path.join(ROOT, "ThermoRoute_results"), "zip", pack)
print("zip:", z)
if IN_COLAB:
    from google.colab import files; files.download(z)

In [ ]:
# ===== Step 6b: final figures (fixed fig A, C, H) at 900 dpi
import importlib, os, glob, zipfile
import matplotlib as mpl
import figs; importlib.reload(figs)
from figs import *                       # helpers, styles and paths from sim/figs.py
mpl.rcParams["savefig.dpi"] = 900

def fig_A():
    from plant import tps
    kp = pd.read_csv(os.path.join(DATA_DIR, "k_profile.csv"))
    un = pd.read_csv(os.path.join(DATA_DIR, "expert_union.csv"))
    p = SOCS[MAIN_COND["soc"]]
    fig, ax = plt.subplots(1, 3, figsize=(DOUBLE, 2.1))
    ax[0].errorbar(kp.k, kp.ppl, yerr=kp.get("ppl_ci", 0 * kp.ppl), fmt="o-", color="#d62728",
                   capsize=2, elinewidth=0.6)
    ax[0].set_ylabel("Perplexity"); ax[0].set_xticks(K_LEVELS)
    tag(ax[0], "a", "Active experts $k$")
    h1, = ax[1].plot(K_LEVELS, [float(tps(p, k, 1.0, "performance")) for k in K_LEVELS], "o-",
                     color="#1f77b4", label="Plain decoding")
    h2, = ax[1].plot(K_LEVELS, [float(tps(p, k, 1.0, "performance", True)) for k in K_LEVELS],
                     "s--", color="#ff7f0e", label="Speculative decoding")
    ax[1].set_ylabel("Decode speed (tok/s)"); ax[1].set_xticks(K_LEVELS)
    tag(ax[1], "b", "Active experts $k$")
    for i, k in enumerate(K_LEVELS):
        u = un[un.k == k].sort_values("window")
        ax[2].plot(u.window, u.union, marker="os^D"[i], ls="-",
                   color=plt.cm.Greys(0.35 + 0.18 * i), label=f"$k={k}$")
    ax[2].set_ylabel("Distinct experts"); ax[2].set_xticks(sorted(un.window.unique()))
    tag(ax[2], "c", "Token window")
    fig.tight_layout(w_pad=1.0)
    h3, l3 = ax[2].get_legend_handles_labels()
    legend_below(fig, [h1, h2] + h3, [h1.get_label(), h2.get_label()] + l3, ncol=6)
    save(fig, "fig_A_k_profile")

def fig_C():
    d = runs()
    fig, ax = plt.subplots(1, 2, figsize=(DOUBLE, 2.3))
    for a, c, l in zip(ax, (MAIN_COND, STRESS_COND), "ab"):
        dc = cond(d, c)
        for m in METHODS:
            r = dc[dc.method == m]; col, mk, _, _ = STYLE[m]
            a.errorbar(r.eff_ppl.mean(), r.j_per_tok.mean(), xerr=ci95(r.eff_ppl),
                       yerr=ci95(r.j_per_tok), fmt=mk, color=col, ms=5, capsize=2,
                       elinewidth=0.6, label=m)
        a.set_ylabel(LABEL["j_per_tok"]); tag(a, l, f"{LABEL['eff_ppl']}  [{cname(c)}]")
        ins = a.inset_axes([0.36, 0.50, 0.40, 0.44])          # zoom on the overlapping points
        xs, ys = [], []
        for m in ["ThermoRoute", "ThermoRoute+Spec", "Oracle"]:
            r = dc[dc.method == m]; col, mk, _, _ = STYLE[m]
            x, y = r.eff_ppl.mean(), r.j_per_tok.mean(); xs.append(x); ys.append(y)
            ins.errorbar(x, y, xerr=ci95(r.eff_ppl), yerr=ci95(r.j_per_tok), fmt=mk, color=col,
                         ms=6 if m != "Oracle" else 8, mfc="none" if m == "Oracle" else col,
                         capsize=2, elinewidth=0.6)
        dx = max(max(xs) - min(xs), 0.004); dy = max(max(ys) - min(ys), 0.002)
        ins.set_xlim(min(xs) - 0.6 * dx, max(xs) + 0.6 * dx)
        ins.set_ylim(min(ys) - 0.6 * dy, max(ys) + 0.6 * dy)
        ins.tick_params(labelsize=6, length=2); ins.grid(True)
        ins.xaxis.set_major_locator(mpl.ticker.MaxNLocator(3))
        ins.yaxis.set_major_locator(mpl.ticker.MaxNLocator(3))
        a.indicate_inset_zoom(ins, edgecolor="0.4", lw=0.5)
    fig.tight_layout(w_pad=1.2)
    h, lab = ax[0].get_legend_handles_labels(); legend_below(fig, h, lab)
    save(fig, "fig_C_pareto")

def fig_H():
    d = runs(); d = d[(d.soc == MAIN_COND["soc"]) & (d.wl == "agent")]
    show = ["ThermoRoute", "ThermoRoute+Spec", "Oracle"]
    cols = [f"frac_k{k}" for k in K_LEVELS]
    shades = plt.cm.Greys(np.linspace(0.15, 0.85, len(K_LEVELS)))
    fig, ax = plt.subplots(figsize=(SINGLE, 2.1))
    pos, labels, p = [], [], 0.0
    for m in show:
        for amb in AMBIENTS:
            fr = d[(d.method == m) & (d.amb == amb)][cols].mean().to_numpy()
            bottom = 0.0
            for j, v in enumerate(fr):
                ax.bar(p, v, 0.8, bottom=bottom, color=shades[j], ec="k", lw=0.3,
                       label=f"$k={K_LEVELS[j]}$" if p == 0 else None)
                bottom += v
            pos.append(p); labels.append(f"{amb:g}"); p += 1
        p += 0.6
    ax.set_xticks(pos); ax.set_xticklabels(labels); ax.tick_params(axis="x", length=0)
    for n, m in enumerate(show):
        ax.text(np.mean(pos[n * 3:(n + 1) * 3]), -0.11, m, ha="center", va="top", fontsize=7,
                transform=ax.get_xaxis_transform())
    ax.set_ylabel("Fraction of active time"); ax.set_ylim(0, 1); ax.grid(axis="x", visible=False)
    ax.set_xlabel("Ambient (°C)", labelpad=11)
    fig.tight_layout()
    h, lab = ax.get_legend_handles_labels(); legend_below(fig, h, lab)
    save(fig, "fig_H_k_occupancy")

figs.fig_A, figs.fig_C, figs.fig_H = fig_A, fig_C, fig_H   # use the fixed versions
figs.make_all()

fig_dir = os.environ["TR_FIG_DIR"]
zip_path = os.path.join(ROOT, "ThermoRoute_figures_900dpi.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(os.path.join(fig_dir, "*.png"))):
        z.write(f, os.path.basename(f))
print("zip:", zip_path)
if IN_COLAB:
    from google.colab import files; files.download(zip_path)